# Appraisal vs Rhet Fig

## Import Packages

In [1]:
import os
import pandas as pd
import re
from ast import literal_eval
from tabulate import tabulate
import random

## Reading and Cleaning Data

### Metaphor Annotations

In [2]:
# load in rhetfig annotation json and save it to a pandas dataframe
file_path = 'RhetFig_May19_2026.json'
rhet_df = pd.read_json(file_path)

In [3]:
# extracting labels from the annotations column
rhet_df['labels'] = rhet_df.apply(lambda row: row.annotations[0]['result'], axis=1)\

# cleaning up file names 
rhet_df['filename'] = rhet_df.apply(lambda row: re.sub(r"^[^_]*-", '', row.file_upload), axis=1)
rhet_df['filename'] = rhet_df.apply(lambda row: re.sub(r"_fixed", '', row.filename[:-4].lower()), axis=1)
rhet_df['filename'] = rhet_df.apply(lambda row: re.sub(r"aboriginal", 'indigenous', row.filename), axis=1)

In [4]:
# creating a dictionary of metaphor labels, where each key is a filename
sarcasm_cnt = 0
rhet_labels = {}
for name in rhet_df['filename'].unique():
    # creating a new dataframe only containing labels corresponding to one file
    new_df = rhet_df[rhet_df['filename']==name][['filename', 'labels']].reset_index()
    # creating a list to save the labels in 
    labels_dic = {}
    for el in new_df['labels'][0]:
        # adding labels to the label list 
        try:
            el['from_name']=='source'
            start = el['value']['start']
            end = el['value']['end']
            for fig in el['value']['taxonomy']:
                if len(fig) > 1: # e.g., ['Irony'. 'Sarcasm'] <- length will always be 1 unless there's a taxonomy
                    if 'Sarcasm' in fig:
                        sarcasm_cnt +=1 
                    label_figure = fig[0] #fig[0] will be Irony, fig[1] will be the subtype
                else: 
                    label_figure = fig[0].replace('Verbal Irony', 'Irony').replace('Synedoche', 'Synecdoche') #replacing typo & holdovers from round 1
                try: 
                    if [start, end] not in labels_dic[label_figure]:
                        labels_dic[label_figure].append([start,end])
                except KeyError: 
                    labels_dic[label_figure] = [[start,end]]
                    # if 'Irony' in label_figure or 'Sarcasm' in label_figure:
                    #     print(labels_dic)
        except KeyError:
            print(el) #these errors correspond to directional labels
    # saving the dic of labels to the dictionary of filenames
    rhet_labels[name] = labels_dic
print(sarcasm_cnt)

{'from_id': 'k1wDguFaUx', 'to_id': 'D6l9c9j9Ec', 'type': 'relation', 'direction': 'right'}
{'from_id': 'llpBo9pgq4', 'to_id': 'L4OqMa4rDB', 'type': 'relation', 'direction': 'right'}
{'from_id': 'nqe6fV-DqG', 'to_id': '8R39NyrMgZ', 'type': 'relation', 'direction': 'right'}
{'from_id': 'N6cnbVpzKN', 'to_id': 'IaDeXnHv-0', 'type': 'relation', 'direction': 'right'}
{'from_id': 'VcV-HgZ-qb', 'to_id': 'CvQ46bQ-pS', 'type': 'relation', 'direction': 'right'}
{'from_id': 'w77jM85eaK', 'to_id': 'ywLTszm2ZE', 'type': 'relation', 'direction': 'right'}
88


In [5]:
len(rhet_labels.keys())

1043

In [6]:
# unique labels
unique_labels = {}
for key in rhet_labels.keys():
    lst = []
    for val in rhet_labels[key].values():
        for el in val:
            if el not in lst:
                lst.append(el)
    unique_labels[key] = lst

In [7]:
# count unique labels
cnt = 0
for v in unique_labels.values():
    cnt+=len(v)
print(cnt)

1978


In [8]:
# list of rhetorical figures
rf_list = []
for file in list(rhet_labels.keys()):
    for rf in rhet_labels[file].keys():
        if rf not in rf_list:
            rf_list.append(rf)

# dictionary where each key is a rhetorical figure and each value is a nested dictionary, 
# where the keys are file names and the values are lists of lists (containing label start and end indices)
rf_dict = {}
for fig in rf_list:
    rf_dict[fig] = {}
    for file in list(rhet_labels.keys()):
        try:
            rf_dict[fig][file] = rhet_labels[file][fig]
        except KeyError:
            rf_dict[fig][file] = []
            pass

In [9]:
# all the labels in a file (counting units with multiple labels multiple times)
all_labels = {}

for k in rhet_labels.keys():
    all_labels[k] = []

for k,v in rhet_labels.items():
    for inds in v.values():
        for ind in inds:
            all_labels[k].append(ind)

In [10]:
# counting total labels
cnt = 0
for v in all_labels.values():
    cnt+=len(v)
print(cnt)

2126


### Appraisal Annotations

In [11]:
def extractor(dic, col, txt):
    '''
    takes dic (a dictionary of dataframes), col (a string corresponding to column name), 
    and txt (a string containing the label type to extract) as input
    returns a dictionary with the same keys as dic, where the values are lists of label lists (e.g., [[1,7],[9,15]])
    '''
    # creating a dictionary of labels, where each key is a filename and each value is a list of labels
    labels_dic = {}
    # looping through all the keys and values in the input dictionary
    for name, df in dic.items():
        # creating an empty list of labels labels
        labels_list = []
        # dropping rows with no labels 
        cleaned_df = df[df[col] != '_'][['indices', col]].dropna()
        # if there are no labels, assigns an empty list
        if len(cleaned_df[col]) == 0:
            labels_dic[name]=labels_list
        else:
            # extracting rows that contain the input txt in the label column
            dic_labels = cleaned_df[cleaned_df[col].str.contains(txt)][col].unique()
            for d in dic_labels:
                cleaned_df_v2 = cleaned_df[cleaned_df[col]==d]
                # adding labels to the list of labels
                new_dic = {}
                new_dic[d] = {}
                # setting unreasonable values
                new_dic[d]['lowest'] = 1000000000
                new_dic[d]['highest'] = -1
                for r in cleaned_df_v2.indices:
                    # splitting the index string into a list
                    ind = [int(x) for x in r.split('-')]
                    # updating lowest index
                    if ind[0] < new_dic[d]['lowest']:
                        new_dic[d]['lowest'] = ind[0]
                    # updating highest index
                    if ind[1] > new_dic[d]['highest']:
                        new_dic[d]['highest'] = ind[1]
                # adding labels to list of labels
                labels_list.append([new_dic[d]['lowest'], new_dic[d]['highest']])
            # assigning list of labels to filename in the label dictionary
            labels_dic[name]=labels_list
    return labels_dic

In [12]:
# uploading central dataframe of file IDs
mapping_df=pd.read_csv('mapping_spreadsheet_2025_08_20.csv')[['MIP', 'Appraisal']]
# dictionary where the MIP name is the key and the Appraisal name is the corresponding value
name_corresp = dict(zip(mapping_df['MIP'], mapping_df['Appraisal']))

In [13]:
# creating a dictionary of appraisal annotation dataframes, where each key is a MIP file name
appraisal_dict = {}

# list of file names in the metaphor annotations
filenames = mapping_df.MIP.unique()

# looping through each file name in the metaphor annotations 
for name in filenames:
    # finding the corresponding folder id for appraisal
    folder_id = name_corresp[name]
    # setting folder path
    path = 'SOCC/annotated/Appraisal/Appraisal_annotations/curation/' + folder_id
    try:
        # loading in appraisal annotations in folders that end with '.txt'
        filename = os.listdir(path + '.txt')[0]
        # reading and saving the annotations to a pandas dataframe
        try:
            df = pd.read_csv(path + '.txt/' + filename, sep = '\t', header = None, 
                             skiprows=5, names=['no.','indices','text','attitude','label','polarity'])
        except:
            df = pd.read_csv(path + '.txt/' + filename, sep = '\t', header = None, 
                             skiprows=6, names=['no.','indices','text','attitude','label','polarity'])
        # saving the dataframe to the appraisal dictionary 
        appraisal_dict[name] = df
    except:
        # loading in appraisal annotations in folders that end with '.tsv'
        filename = os.listdir(path + '.tsv')[0]
        # reading and saving the annotations to a pandas dataframe
        try:
            df = pd.read_csv(path + '.tsv/' + filename, sep = '\t', header = None, skiprows=5, 
                             names=['no.','indices','text','attitude','label','polarity'])
        except:
            df = pd.read_csv(path + '.tsv/' + filename, sep = '\t', header = None, skiprows=6, 
                             names=['no.','indices','text','attitude','label','polarity'])
        # saving the dataframe to the appraisal dictionary
        appraisal_dict[name] = df

In [14]:
# cleaning labels in the label and attitude columns by removing the annotation before the '|'
for filename in appraisal_dict.keys():
    appraisal_dict[filename]['label_final']=appraisal_dict[filename].apply(lambda row: str(row.label).split('|')[-1], axis=1)
    appraisal_dict[filename]['attitude_final']=appraisal_dict[filename].apply(lambda row: str(row.attitude).split('|')[-1], axis=1)

In [15]:
# confirming that files all match
print(len(unique_labels.keys()), len(appraisal_dict.keys()), 
      len(set(list(unique_labels.keys()) + list(appraisal_dict.keys()))),
      set(list(unique_labels.keys())) - set(list(appraisal_dict.keys())),
      set(list(appraisal_dict.keys())) - set(list(unique_labels.keys())))

1043 1043 1043 set() set()


In [16]:
# a dictionary where every key is a file name and every value is a list of the character indices icluded in the negative labels 
neg_labels = extractor(appraisal_dict, 'label_final', 'neg')
# a dictionary where every key is a file name and every value is a list of the character indices included in the positive labels 
pos_labels = extractor(appraisal_dict, 'label_final', 'pos')
# a dictionary where every key is a file name and every value is a list of the character indices included in the Appreciation labels 
appr_labels = extractor(appraisal_dict, 'attitude_final', 'Appreciation')
# a dictionary where every key is a file name and every value is a list of the character indices included in the Judgement labels 
judg_labels = extractor(appraisal_dict, 'attitude_final', 'Judgment') # note spelling
# a dictionary where every key is a file name and every value is a list of the character indices included in the Affect labels 
aff_labels = extractor(appraisal_dict, 'attitude_final', 'Affect')

## Comparison 
### what are we measuring? how?
We are deriving ten metrics to measure the overlap between rhetorical figures (RF) and Appraisal and, within Appraisal, between negative evaluation and RF. The first five take the RF annotations as their starting point and calculate how many of those were also labelled in the Appraisal annotations, broken down into subcategories of Appraisal: 
1. Percentage of RF units that are labelled as Affect 
2. Percentage of RF units that are labelled as Judgement  
3. Percentage of RF units that are labelled as Appreciation  
4. Percentage of RF units that are labelled as positive 
5. Percentage of RF units that are labelled as negative 
The next five metrics take an Appraisal label as their starting point and calculate what percentage of those have RF labels:
1. Percentage of Affect units that are labelled as RF  
2. Percentage of Judgement units that are labelled as RF 
3. Percentage of Appreciation units that are labelled as RF 
4. Percentage of positive units that are labelled as RF 
5. Percentage of negative units that are labelled as negative 
For (1), we first go through each label in the RF annotations, and check if 30% of the characters show up in a Affect label. If they do, we say that that RF unit is labelled as Affect. We then calculate the percentage of Affect RF units by dividing the number of RF that are labelled as Affect by the number of units labelled as RF overall. The other metrics follow a similar methodology. 

In [17]:
def overlap_calculator_helper(dic1, dic2):
    '''
    returns a list containing the percentage and number of dic1 units that are labelled in dic2
    '''
    # setting counter for number of labels overall
    dic1_total_labels, dic2_total_labels = 0, 0
    
    # number of labels in each dic that overlap (e.g., if dic1 has labels [1,3] and [4,9] for a file 
    # and dic2 has a label [1,9] for the same file, then dic1 has two overlapping labels with dic2 
    # and dic2 has 1 overlapping label with dic1)
    overall_overlap_cnt_dic1, overall_overlap_cnt_dic2 = 0, 0
    
    for file in list(dic1.keys()):
        # initializing overlap count for each file in each dictionary; setting it to 0
        overlap_cnt_dic1, overlap_cnt_dic2 = 0, 0

        # loops through each label list in the list of list of labels corresponding to each key in the dictionary 
        for label_dic1 in dic1[file]:
                # creating a set of the characters contained in the indices (e.g., [1,5] -> [1,2,3,4,5])
                label_dic1_characters = set(list(range(label_dic1[0],label_dic1[1]+1)))
                # loops through the labels in the second dictionary for the same key
                for label_dic2 in dic2[file]:
                    # creating a set of the characters contained in the indices (e.g., [1,5] -> [1,2,3,4,5])
                    label_dic2_characters = set(list(range(label_dic2[0],label_dic2[1]+1)))
                    # variable representing the characters included in both labels (intersection)
                    overlap = label_dic1_characters & label_dic2_characters
                    # variable representing the elements included in either label (union)
                    universe = label_dic2_characters | label_dic2_characters
                    
                    # calculating overlap % if more than one element exists in both labels 
                    # setting overlap to > 1 accounts for punctuation, etc.
                    if len(overlap) > 1:
                        # the percentage of the first label that is included in the second label
                        result_dic1 = float(len(overlap)) / len(label_dic1_characters) * 100
                        # if at least 30% of the first label is included in the second label, it counts as overlap for the first label
                        if result_dic1 >= 30:
                            overlap_cnt_dic1+=1
                            overall_overlap_cnt_dic1+=1
                            break

        # adding to counter of total labels for dic1
        dic1_total_labels = dic1_total_labels + len(dic1[file])
    # percentage of dic1 units that are labelled in dic2
    return [overall_overlap_cnt_dic1/dic1_total_labels * 100,overall_overlap_cnt_dic1]
    
def overlap_calculator(dic1, dic2):
    '''
    returns a dictionary containing the percentage of dic1 units that are labelled in dic2 and 
    the percentage of dic2 units that are labelled in dic1, the total number of dic1 units labelled in dic2,
    and the total number of dic2 units labelled in dic1
    '''
    return {'percentage of dic1 units that are labelled in dic2': overlap_calculator_helper(dic1, dic2)[0],
            'percentage of dic2 units that are labelled in dic1': overlap_calculator_helper(dic2, dic1)[0],
            'overall_overlap_cnt_dic1': overlap_calculator_helper(dic1, dic2)[1],
            'overall_overlap_cnt_dic2': overlap_calculator_helper(dic2, dic1)[1]}

In [18]:
overlap_percentage_pos = overlap_calculator(pos_labels, all_labels).values()
overlap_percentage_neg = overlap_calculator(neg_labels, all_labels).values()
overlap_percentage_appr = overlap_calculator(appr_labels, all_labels).values()
overlap_percentage_aff = overlap_calculator(aff_labels, all_labels).values()
overlap_percentage_judg = overlap_calculator(judg_labels, all_labels).values()

print(f'percentage of X labels that are labelled as rhetorical figures')
print(tabulate([['Positive', list(overlap_percentage_pos)[2], round(list(overlap_percentage_pos)[0],2)], 
                ['Negative', list(overlap_percentage_neg)[2], round(list(overlap_percentage_neg)[0],2)],
                ['Appreciation', list(overlap_percentage_appr)[2], round(list(overlap_percentage_appr)[0],2)], 
                ['Affect', list(overlap_percentage_aff)[2], round(list(overlap_percentage_aff)[0],2)],
                ['Judgement', list(overlap_percentage_judg)[2], round(list(overlap_percentage_judg)[0],2)]], headers=['X', 'Count', '%']))

percentage of X labels that are labelled as rhetorical figures
X               Count      %
------------  -------  -----
Positive          138  21.5
Negative          735  36.95
Appreciation      361  27.1
Affect             16  20.51
Judgement         502  39.13


In [19]:
overlap_percentage_pos = overlap_calculator(pos_labels, unique_labels).values()
overlap_percentage_neg = overlap_calculator(neg_labels, unique_labels).values()
overlap_percentage_appr = overlap_calculator(appr_labels, unique_labels).values()
overlap_percentage_aff = overlap_calculator(aff_labels, unique_labels).values()
overlap_percentage_judg = overlap_calculator(judg_labels, unique_labels).values()

print(f'percentage of rhetorical figure units that are labelled as X')
print(tabulate([['Positive', list(overlap_percentage_pos)[3], round(list(overlap_percentage_pos)[1],2)], 
                ['Negative', list(overlap_percentage_neg)[3], round(list(overlap_percentage_neg)[1],2)],
                ['Appreciation', list(overlap_percentage_appr)[3], round(list(overlap_percentage_appr)[1],2)], 
                ['Affect', list(overlap_percentage_aff)[3], round(list(overlap_percentage_aff)[1],2)],
                ['Judgement', list(overlap_percentage_judg)[3], round(list(overlap_percentage_judg)[1],2)]], headers=['X', 'Count', '%']))

percentage of rhetorical figure units that are labelled as X
X               Count      %
------------  -------  -----
Positive          116   5.86
Negative          664  33.57
Appreciation      315  15.93
Affect              9   0.46
Judgement         481  24.32


We also measure the percentage of RF labels that overlap with Attitude labels and the percentage of Attitude labels that overlap with RF labels.

In [20]:
# measuring the percentage of RF labels that overlap with Attitude labels and the percentage of Attitude labels that overlap with RF labels

# a dictionary where every key is a file name and every value is a list of the character indices included in the Appreciation labels 
appr_labels = extractor(appraisal_dict, 'attitude_final', 'Appreciation')
# a dictionary where every key is a file name and every value is a list of the character indices included in the Judgement labels 
judg_labels = extractor(appraisal_dict, 'attitude_final', 'Judgment') # note spelling
# a dictionary where every key is a file name and every value is a list of the character indices included in the Affect labels 
aff_labels = extractor(appraisal_dict, 'attitude_final', 'Affect')

# dictionary of Attitude labels, based on Appreciation, Judgement, and Affect  
att_labels = {}
for p in list(appr_labels.keys()):
    att_labels[p] = appr_labels[p].copy()
for j in list(judg_labels.keys()):
    if judg_labels[j] != []:
        att_labels[j] = att_labels[j] + judg_labels[j]
for f in list(aff_labels.keys()):
    if aff_labels[f] != []:
        att_labels[f] = att_labels[f] + aff_labels[f]

overlap_calculator(att_labels, unique_labels)

{'percentage of dic1 units that are labelled in dic2': 32.64017823988117,
 'percentage of dic2 units that are labelled in dic1': 38.119312436804854,
 'overall_overlap_cnt_dic1': 879,
 'overall_overlap_cnt_dic2': 754}

Breaking down our analysis by RF type:

In [21]:
# how much Attitude overlaps with each RF
for rf in rf_dict.keys():
    over = overlap_calculator(att_labels, rf_dict[rf])['percentage of dic1 units that are labelled in dic2']
    print(rf, round(over,2))

Antithesis 3.34
Synecdoche 0.11
Simile 0.37
Personification 5.61
Metonymy 2.04
Rhetorical Question 10.21
Irony 5.98
Hyperbole 5.72
Puns/Wordplay 0.71
Allusion 1.89
Euphemism 0.89
Litotes 0.15


In [22]:
# breakdown of Appraisal label overlap with RF by RF type
results_dict = {}
for fig in rf_dict.keys():
    # print(fig)
    results_dict[fig] = {}
    labels = rf_dict[fig]
    overlap_percentage_pos = overlap_calculator(pos_labels, labels).values()
    results_dict[fig]['pos'] = list(overlap_percentage_pos)[:4]
    # print(f'percentage of positive units that are labelled as {fig}:', list(overlap_percentage_pos)[0])
    # print(f'percentage of {fig} units that are labelled as positive:', list(overlap_percentage_pos)[1])
    
    overlap_percentage_neg = overlap_calculator(neg_labels, labels).values()
    results_dict[fig]['neg'] = list(overlap_percentage_neg)[:4]
    # print(f'percentage of negative units that are labelled as {fig}:', list(overlap_percentage_neg)[0])
    # print(f'percentage of {fig} units that are labelled as negative:', list(overlap_percentage_neg)[1])
    
    overlap_percentage_appr = overlap_calculator(appr_labels, labels).values()
    results_dict[fig]['appr'] = list(overlap_percentage_appr)[:4]
    # print(f'percentage of Appreciation units that are labelled as {fig}:', list(overlap_percentage_appr)[0])
    # print(f'percentage of {fig} units that are labelled as Appreciation:', list(overlap_percentage_appr)[1])
    
    overlap_percentage_aff = overlap_calculator(aff_labels, labels).values()
    results_dict[fig]['aff'] = list(overlap_percentage_aff)[:4]
    # print(f'percentage of Affect units that are labelled as {fig}:', list(overlap_percentage_aff)[0])
    # print(f'percentage of {fig} units that are labelled as Affect:', list(overlap_percentage_aff)[1])

    overlap_percentage_judg = overlap_calculator(judg_labels, labels).values()
    results_dict[fig]['judg'] = list(overlap_percentage_judg)[:4]
    # print(f'percentage of Judgement units that are labelled as {fig}:', list(overlap_percentage_judg)[0])
    # print(f'percentage of {fig} units that are labelled as Judgement:', list(overlap_percentage_judg)[1])

for fig in results_dict.keys():
    print(f'\npercentage of X units that are labelled as {fig}')

    
    print(tabulate([['Positive', results_dict[fig]['pos'][2], round(results_dict[fig]['pos'][0],2)], 
                    ['Negative', results_dict[fig]['neg'][2], round(results_dict[fig]['neg'][0],2)],
                   ['Appreciation', results_dict[fig]['appr'][2], round(results_dict[fig]['appr'][0],2)], 
                    ['Affect', results_dict[fig]['aff'][2], round(results_dict[fig]['aff'][0],2)],
                   ['Judgement', results_dict[fig]['judg'][2], round(results_dict[fig]['judg'][0],2)]], headers=['X', 'Count', '%']))


percentage of X units that are labelled as Antithesis
X               Count     %
------------  -------  ----
Positive           32  4.98
Negative           59  2.97
Appreciation       41  3.08
Affect              1  1.28
Judgement          48  3.74

percentage of X units that are labelled as Synecdoche
X               Count     %
------------  -------  ----
Positive            0  0
Negative            3  0.15
Appreciation        2  0.15
Affect              0  0
Judgement           1  0.08

percentage of X units that are labelled as Simile
X               Count     %
------------  -------  ----
Positive            1  0.16
Negative            9  0.45
Appreciation        5  0.38
Affect              0  0
Judgement           5  0.39

percentage of X units that are labelled as Personification
X               Count     %
------------  -------  ----
Positive           37  5.76
Negative          115  5.78
Appreciation       91  6.83
Affect              1  1.28
Judgement          59  4.6

perc

In [23]:
# breakdown of RF overlap with Appraisal by RF type
results_dict = {}
for fig in rf_dict.keys():
    results_dict[fig] = {}
    labels = rf_dict[fig]
    overlap_percentage_pos = overlap_calculator(pos_labels, labels).values()
    overlap_percentage_neg = overlap_calculator(neg_labels, labels).values()   
    overlap_percentage_appr = overlap_calculator(appr_labels, labels).values()
    overlap_percentage_aff = overlap_calculator(aff_labels, labels).values()
    overlap_percentage_judg = overlap_calculator(judg_labels, labels).values()

    print(f'percentage of {fig} units that are labelled as X')
    print(tabulate([['Positive', list(overlap_percentage_pos)[3], round(list(overlap_percentage_pos)[1],2)], 
                    ['Negative', list(overlap_percentage_neg)[3], round(list(overlap_percentage_neg)[1],2)],
                    ['Appreciation', list(overlap_percentage_appr)[3], round(list(overlap_percentage_appr)[1],2)], 
                    ['Affect', list(overlap_percentage_aff)[3], round(list(overlap_percentage_aff)[1],2)],
                    ['Judgement', list(overlap_percentage_judg)[3], round(list(overlap_percentage_judg)[1],2)]], headers=['X', 'Count', '%']))

percentage of Antithesis units that are labelled as X
X               Count      %
------------  -------  -----
Positive           14  10.94
Negative           33  25.78
Appreciation       19  14.84
Affect              0   0
Judgement          27  21.09
percentage of Synecdoche units that are labelled as X
X               Count      %
------------  -------  -----
Positive            2   9.52
Negative            9  42.86
Appreciation        7  33.33
Affect              0   0
Judgement           3  14.29
percentage of Simile units that are labelled as X
X               Count      %
------------  -------  -----
Positive            1   2.44
Negative            8  19.51
Appreciation        6  14.63
Affect              0   0
Judgement           6  14.63
percentage of Personification units that are labelled as X
X               Count      %
------------  -------  -----
Positive           34   8.88
Negative          108  28.2
Appreciation       81  21.15
Affect              1   0.26
Judgement 

In [25]:
# dictionary where key is a rhetorical figure and value is the total count of that figure
rf_count = {}
for key in rf_dict.keys():
    dic1 = rf_dict[key]
    dic1_total_labels = 0
    
    for file in list(dic1.keys()):
        # adding to counter of total labels for dic1
        dic1_total_labels = dic1_total_labels + len(dic1[file])
    
    rf_count[key] = dic1_total_labels

In [26]:
# dataframe with a breakdown of RF overlap with Appraisal by RF type, including total count for that figure, 
# count of that figure overlapping with Attitude, count of that figure overlapping with positive/negative evaluation,
# and percentages of overlap with Attitude and positive/negative evaluation
breakdown_df = df = pd.DataFrame(columns = ['RF Label' , 'RF Count', 'Appraisal Label', 'Attitude Count' , 'Attitude %', 'Polarity Label',
                                              'Polarity Count', 'Polarity %']) 

results_dict = {}
for fig in rf_dict.keys():
    results_dict[fig] = {}
    labels = rf_dict[fig]
    overlap_percentage_pos = overlap_calculator(pos_labels, labels).values()
    overlap_percentage_neg = overlap_calculator(neg_labels, labels).values()   
    overlap_percentage_att = overlap_calculator(att_labels, labels).values()

    fig_df = pd.DataFrame([[fig, rf_count[fig], 'Attitude', list(overlap_percentage_att)[3], 
                            round(list(overlap_percentage_att)[1],2), 
                           'Positive', list(overlap_percentage_pos)[3], round(list(overlap_percentage_pos)[1],2)], 
                           [fig, rf_count[fig], 'Attitude', list(overlap_percentage_att)[3], 
                            round(list(overlap_percentage_att)[1],2),
                          'Negative', list(overlap_percentage_neg)[3], round(list(overlap_percentage_neg)[1],2)]], 
                          columns=['RF Label' , 'RF Count', 'Appraisal Label', 'Attitude Count' , 'Attitude %', 'Polarity Label', 
                                   'Polarity Count', 'Polarity %'])
    breakdown_df = pd.concat([breakdown_df, fig_df])

breakdown_df.sort_values(by=['RF Count', 'Polarity Label'], ascending=False)
#breakdown_df.sort_values(by=['RF Count', 'Polarity Label'], ascending=False).to_csv('rhetfig_appraisal_v2.csv')

C:\Users\romha\AppData\Local\Temp\ipykernel_42868\295409533.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  breakdown_df = pd.concat([breakdown_df, fig_df])


,RF Label,RF Count,Appraisal Label,Attitude Count,Attitude %,Polarity Label,Polarity Count,Polarity %
0,Metonymy,522,Attitude,141,27.01,Positive,24,4.60
1,Metonymy,522,Attitude,141,27.01,Negative,118,22.61
0,Personification,383,Attitude,139,36.29,Positive,34,8.88
1,Personification,383,Attitude,139,36.29,Negative,108,28.20
0,Hyperbole,349,Attitude,133,38.11,Positive,14,4.01
1,Hyperbole,349,Attitude,133,38.11,Negative,122,34.96
0,Rhetorical Question,331,Attitude,178,53.78,Positive,11,3.32
1,Rhetorical Question,331,Attitude,178,53.78,Negative,171,51.66
0,Irony,150,Attitude,85,56.67,Positive,6,4.00
1,Irony,150,Attitude,85,56.67,Negative,83,55.33


## Overlap Examples

In [27]:
# selecting 50 files at random to check for overlap between metaphor and appraisal
random.seed(10)
example_files = random.sample(list(filenames), 50)

In [28]:
def overlap_examples(dic1, dic2, examples):
    for file in examples:
        for label_dic1 in dic1[file]:
            # creating a set of the characters contained in the indices (e.g., [1,5] -> [1,2,3,4,5])
            label_dic1_characters = set(list(range(label_dic1[0],label_dic1[1]+1)))
            # loops through the labels in the second dictionary for the same key
            for label_dic2 in dic2[file]:
                # creating a set of the characters contained in the indices (e.g., [1,5] -> [1,2,3,4,5])
                label_dic2_characters = set(list(range(label_dic2[0],label_dic2[1]+1)))
                # variable representing the characters included in both labels (intersection)
                overlap = label_dic1_characters & label_dic2_characters
                # variable representing the elements included in either label (union)
                universe = label_dic2_characters | label_dic2_characters
                
                # calculating overlap % if more than one element exists in both labels 
                # setting overlap to > 1 accounts for punctuation, etc.
                if len(overlap) > 1:
                    # the percentage of the first label that is included in the second label
                    result_dic1 = float(len(overlap)) / len(label_dic1_characters) * 100
                    # the percentage of the second label that is included in the first label
                    result_dic2 = float(len(overlap)) / len(label_dic2_characters) * 100
                    # if at least 30% of one label is included in the other label, it counts as overlap
                    if result_dic1 >= 30 or result_dic2 >= 30:
                        # increasing counters by 1
                        text = rhet_df[rhet_df['filename']==file].reset_index().data[0]['text']
                        print('file name: {}\n text: {}\n metaphor label: {}\n appraisal label: {}\n overlap: {}'.format(file, 
                              text,
                              text[label_dic1[0]:label_dic1[1]],
                              text[label_dic2[0]:label_dic2[1]],
                              text[list(overlap)[0]: list(overlap)[-1]]))

In [29]:
print("Irony and negative overlap examples:")
overlap_examples(rf_dict['Irony'], neg_labels, example_files)

Irony and negative overlap examples:
file name: watch_56
 text: I, on the other hand, feel lost when I'm not wearing my watch. I'm not always in front of my computer or in my car or in a room with a clock; sometimes I'm (hard to believe, I know) outside, doing yardwork, or at the local garden centre, or at a friend's house with a clock not readily apparent. Indeed, the last scenario played out a week or so ago, at a meeting, and someone asked what time it was. I glanced at my wrist and had the answer while everyone else was still reaching for their phone.
 metaphor label: (hard to believe, I know)
 appraisal label: hard to believe, I know
 overlap: hard to believe, I know
file name: belgium_120
 text: it's them Christian European hordes, doncha know; they're an inherently fractious and irrational people -- as millenia of ethnic and sectarian rivalry and bloodshed will attest to.; - )http://www.telegraph.co.uk/news/worldnews/europe/ireland/12203336/The-bloody-legacy-of-the-Easter-Rising